# Module 8 — Functions

**Part II: Core Skills** · Level: Beginner → Intermediate · Prerequisites: Module 7

## What you'll learn

- Defining functions and returning values
- Every parameter style: positional, keyword, default, `*args`, `**kwargs`, keyword-only
- Docstrings and annotations
- Lambdas and higher-order functions
- Recursion and base cases
- Useful built-in functions

---

## 8.1 Defining and calling

Functions package reusable logic under a name. `return` sends a value back to the caller; a function with no `return` gives back `None`.

In [ ]:
def add(a, b):
    return a + b

print(add(2, 3))                 # 5

def square(x):
    return x * x

print(square(5))                 # 25

def say_hello():
    print("hello")               # no return statement

result = say_hello()
print(result)                    # None

Functions can return several values — really a single tuple that you unpack:

In [ ]:
def min_max(values):
    return min(values), max(values)

low, high = min_max([4, 9, 1, 7])
print(low, high)                 # 1 9

## 8.2 Parameters and arguments

*Parameters* are the names in the definition; *arguments* are the values you pass.

In [ ]:
def run(job, workers):
    print(job, workers)

run("backup", 4)                 # positional: matched by order
run(workers=4, job="backup")     # keyword: matched by name, any order

### Default arguments

In [ ]:
def run(job="backup", workers=2):
    print(job, workers)

run()                            # backup 2
run("sync")                      # sync 2
run(workers=8)                   # backup 8

Parameters with defaults must come after those without.

### `*args` and `**kwargs`

`*args` collects extra positional arguments into a **tuple**; `**kwargs` collects extra keyword arguments into a **dict**.

In [ ]:
def show(*values):
    print(values)

show("a", "b", "c")              # ('a', 'b', 'c')

def options(**opts):
    print(opts)

options(seed=42, shots=1024)     # {'seed': 42, 'shots': 1024}

The same stars work at the *call site* to unpack:

In [ ]:
def configure(shots, seed):
    return shots, seed

args = (100, 7)
kwargs = {"shots": 100, "seed": 7}
print(configure(*args), configure(**kwargs))

### Keyword-only and positional-only

A bare `*` forces later parameters to be passed by name; a `/` forces earlier ones to be positional.

In [ ]:
def make_report(title, *, include_total=False, **metadata):
    """Build a small report string.

    Args:
        title: Report heading.
        include_total: Append a TOTAL line when True (keyword-only).
        **metadata: Extra labelled values appended to the report.
    """
    lines = [title.upper()]
    for key, value in metadata.items():
        lines.append(f"{key}: {value}")
    if include_total:
        lines.append(f"TOTAL FIELDS: {len(metadata)}")
    return "\n".join(lines)

print(make_report("Q3", include_total=True, author="sam", region="west"))

def distance(x, y, /, scale=1):       # x and y must be positional
    return (x ** 2 + y ** 2) ** 0.5 * scale

print(distance(3, 4), distance(3, 4, scale=2))

## 8.3 Docstrings and annotations

A docstring (first statement in the body) documents purpose, parameters, and return value. **Annotations** state the intended types; Python does not enforce them, but editors and checkers use them (more in Module 15).

In [ ]:
def measure(items: int, label: str = "run") -> str:
    """Return a one-line summary of a run.

    Args:
        items: How many items were processed.
        label: A name for the run.

    Returns:
        A formatted summary string.
    """
    return f"{label}: processed {items} items"

print(measure(512))
print(measure.__annotations__)
help(measure)

## 8.4 Guard clauses and validation

Check bad input first and exit early; the "happy path" then stays unindented.

In [ ]:
def average(values: list[float]) -> float:
    if not values:
        raise ValueError("average() needs at least one value")
    return sum(values) / len(values)

print(average([2, 4, 6]))

## 8.5 Lambda functions

A `lambda` is a small anonymous function limited to a single expression. It is most useful as a throwaway argument.

In [ ]:
double = lambda n: n * 2           # fine to demonstrate; prefer def for named functions
print(double(5))

values = [3, 1, 2]
print(sorted(values, key=lambda x: -x))       # [3, 2, 1]

counts = {"00": 420, "01": 25, "10": 35, "11": 520}
by_count = sorted(counts.items(), key=lambda item: item[1], reverse=True)
print(by_count)

## 8.6 Higher-order functions

Functions are ordinary objects: they can be passed as arguments, returned, and stored in data structures.

In [ ]:
def apply(func, x):
    return func(x)

print(apply(lambda n: n * 2, 5))     # 10
print(apply(str.upper, "shout"))     # SHOUT

operations = {
    "add": lambda a, b: a + b,
    "mul": lambda a, b: a * b,
}
print(operations["mul"](6, 7))       # 42

`map`, `filter`, and `sorted(key=...)` take functions too; they are covered in Module 11.

## 8.7 Recursion and base cases

A **recursive** function calls itself on a smaller version of the problem. Every recursive function needs a **base case** that stops the calls, or Python raises `RecursionError` (the default limit is about 1000 frames).

In [ ]:
def countdown(n):
    if n == 0:             # base case
        return
    print(n)
    countdown(n - 1)       # recursive case

countdown(3)

def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))        # 120

def flatten(nested):
    """Flatten arbitrarily nested lists."""
    flat = []
    for item in nested:
        if isinstance(item, list):
            flat.extend(flatten(item))
        else:
            flat.append(item)
    return flat

print(flatten([1, [2, [3, 4]], 5]))   # [1, 2, 3, 4, 5]

Many recursive problems are also solvable with a loop; recursion shines on tree-like data. Module 17 shows how to make naive recursion fast with caching.

## 8.8 Useful built-in functions

In [ ]:
data = [2, 5, 3]
print(len("python"), max(data), min(data), sum(data))
print(abs(-7), round(3.14159, 2), pow(2, 10))
print(sorted(data), list(reversed(data)))
print(any(x > 4 for x in data), all(x > 4 for x in data))
print(isinstance(3, int), callable(len))
print(list(range(0, 10, 3)), type(data).__name__)
print(dir(data)[-3:])      # peek at available attributes

## Common pitfalls

- Mutable default arguments (`def f(x, items=[])`) — the default is created once and shared. Use `None` and create the list inside. Explained in Module 10.
- Forgetting `return`, which silently gives you `None`.
- Printing instead of returning: `print` is for humans, `return` is for other code.
- Overusing `lambda` for anything longer than a trivial expression.
- Recursion without a reachable base case.

## Practice

1. Write `clamp(value, low, high)` that limits a value to a range.
2. Write `summarize(*numbers)` returning a dict with `count`, `total`, and `mean`.
3. Write a recursive `power(base, exp)` and compare it with `**`.